# TENTACLES — Official implementation verification

Pinned R source, dependency validation, API inspection, and preprocessing example.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import re
import shutil
import subprocess

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")

candidates = []
if os.getenv("CMPS355_ROOT"):
    candidates.append(Path(os.environ["CMPS355_ROOT"]).expanduser())
candidates.extend([Path("/content/drive/MyDrive/CMPS355_Research"), Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates if (p / ".cmps355_root").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Project root CMPS355_Research was not found")
SCRIPT = ROOT / "r" / "verify_tentacles.R"
if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)
RSCRIPT = shutil.which("Rscript")
GIT = shutil.which("git")
if not RSCRIPT or not GIT:
    raise RuntimeError("Both Rscript and git are required")
R_LIB = Path("/content/cmps355_r_lib") if drive is not None else ROOT / "runs" / "r_library"
R_LIB.mkdir(parents=True, exist_ok=True)
RUN_DIR = ROOT / "runs" / "official_r"
RUN_DIR.mkdir(parents=True, exist_ok=True)
LOCK_PATH = ROOT / "configs" / "tentacles.lock.json"
REPO = "https://github.com/Giomu/TENTACLES.git"

if LOCK_PATH.is_file():
    SHA = json.loads(LOCK_PATH.read_text(encoding="utf-8"))["source_commit"]
else:
    refs = subprocess.check_output([GIT, "ls-remote", REPO, "HEAD"], text=True, timeout=90).strip()
    SHA = refs.split()[0]
if not re.fullmatch(r"[a-f0-9]{40}", SHA):
    raise RuntimeError("A valid 40-character source SHA was not obtained")

SOURCE_DIR = Path("/content/cmps355_tentacles_source") if drive is not None else RUN_DIR / "tentacles_source"
if SOURCE_DIR.exists():
    previous = subprocess.run([GIT, "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], capture_output=True, text=True)
    if previous.returncode != 0 or previous.stdout.strip() != SHA:
        shutil.rmtree(SOURCE_DIR)
if not SOURCE_DIR.exists():
    SOURCE_DIR.mkdir(parents=True)
    subprocess.run([GIT, "-C", str(SOURCE_DIR), "init", "-q"], check=True)
    subprocess.run([GIT, "-C", str(SOURCE_DIR), "remote", "add", "origin", REPO], check=True)
    subprocess.run([GIT, "-C", str(SOURCE_DIR), "fetch", "--depth=1", "-q", "origin", SHA], check=True, timeout=240)
    subprocess.run([GIT, "-C", str(SOURCE_DIR), "checkout", "--detach", "-q", "FETCH_HEAD"], check=True)
resolved = subprocess.check_output([GIT, "-C", str(SOURCE_DIR), "rev-parse", "HEAD"], text=True).strip()
if resolved != SHA or not (SOURCE_DIR / "DESCRIPTION").is_file():
    raise RuntimeError("Source checkout does not match the pinned revision")
print("Project:", ROOT)
print("Source commit:", SHA)
print("R:", subprocess.check_output([RSCRIPT, "--version"], stderr=subprocess.STDOUT, text=True).strip())
print("R script SHA-256:", hashlib.sha256(SCRIPT.read_bytes()).hexdigest())

Mounted at /content/drive
Project: /content/drive/MyDrive/CMPS355_Research
Source commit: 069a26271e3885e015df25baa40a7a1140827d5f
R: Rscript (R) version 4.6.1 (2026-06-24)
R script SHA-256: 03243bd1b77a0b94901d81c44704aa69f7ece8255786030eae96d07426deb10f


In [2]:
import pathlib
import platform
import tarfile
import tempfile
import time

R_FULL_VERSION = subprocess.check_output(
    [RSCRIPT, "--vanilla", "-e", "cat(as.character(getRversion()))"], text=True
).strip()
OS_ID = platform.freedesktop_os_release().get("ID", "linux")
OS_VERSION = platform.freedesktop_os_release().get("VERSION_ID", "unknown")
ARCH = platform.machine()
CACHE_DIR = ROOT / "runs" / "environment_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_KEY = f"r{R_FULL_VERSION}_{OS_ID}{OS_VERSION}_{ARCH}"
CACHE_ARCHIVE = CACHE_DIR / f"{CACHE_KEY}.tar.gz"
CACHE_MANIFEST = CACHE_DIR / f"{CACHE_KEY}.json"

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def completed_packages():
    return [p for p in R_LIB.iterdir() if p.is_dir() and not p.name.startswith("00LOCK")]

def save_r_cache():
    if not completed_packages():
        return False
    with tempfile.TemporaryDirectory(dir="/content" if Path("/content").exists() else None) as temp:
        output = Path(temp) / "r_library.tar.gz"
        with tarfile.open(output, mode="w:gz", compresslevel=1) as archive:
            for package in sorted(completed_packages()):
                archive.add(package, arcname=f"r_library/{package.name}")
        digest = sha256_file(output)
        staged = CACHE_ARCHIVE.with_name(CACHE_ARCHIVE.name + ".tmp")
        shutil.copyfile(output, staged)
        staged.replace(CACHE_ARCHIVE)
    manifest = {
        "r_version": R_FULL_VERSION,
        "os": OS_ID,
        "os_version": OS_VERSION,
        "arch": ARCH,
        "sha256": digest,
        "package_directories": len(completed_packages()),
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    staged_manifest = CACHE_MANIFEST.with_name(CACHE_MANIFEST.name + ".tmp")
    staged_manifest.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    staged_manifest.replace(CACHE_MANIFEST)
    print(f"Cached {manifest['package_directories']} package directories", flush=True)
    return True

def restore_r_cache():
    if not CACHE_ARCHIVE.is_file() or not CACHE_MANIFEST.is_file():
        return False
    manifest = json.loads(CACHE_MANIFEST.read_text(encoding="utf-8"))
    for key, expected in (("r_version", R_FULL_VERSION), ("os", OS_ID),
                          ("os_version", OS_VERSION), ("arch", ARCH)):
        if manifest.get(key) != expected:
            print("R cache environment mismatch; ignoring cache")
            return False
    if sha256_file(CACHE_ARCHIVE) != manifest.get("sha256"):
        raise RuntimeError("R cache archive checksum mismatch")
    with tempfile.TemporaryDirectory(dir="/content" if Path("/content").exists() else None) as temp:
        with tarfile.open(CACHE_ARCHIVE, mode="r:gz") as archive:
            if any(not item.name.startswith("r_library/") or ".." in pathlib.PurePosixPath(item.name).parts for item in archive.getmembers()):
                raise RuntimeError("Unexpected cache archive layout")
            archive.extractall(temp, filter="data")
        cached = Path(temp) / "r_library"
        count = 0
        if cached.is_dir():
            for package in cached.iterdir():
                target = R_LIB / package.name
                if not target.exists():
                    shutil.copytree(package, target)
                    count += 1
    print(f"Restored {count} package directories from Drive cache")
    return True

RESTORED_R_CACHE = restore_r_cache()
print("R cache:", CACHE_ARCHIVE.relative_to(ROOT))
print("R packages available:", len(completed_packages()))


R cache: runs/environment_cache/r4.6.1_ubuntu24.04_x86_64.tar.gz
R packages available: 0


In [3]:
from pathlib import Path
import os
import shutil
import subprocess
import tempfile

RUST_TOOLCHAIN = "1.90.0"
cargo_bin = Path.home() / ".cargo" / "bin"
os.environ["PATH"] = f"{cargo_bin}:{os.environ['PATH']}"
os.environ["CARGO_BUILD_JOBS"] = "2"

def rust_version():
    executable = shutil.which("rustc")
    if executable is None:
        return None
    result = subprocess.run([executable, "--version"], capture_output=True, text=True, check=True)
    return result.stdout.strip()

if shutil.which("rustup") is None:
    with tempfile.TemporaryDirectory() as temporary:
        installer = Path(temporary) / "rustup-init.sh"
        subprocess.run(["curl", "--fail", "--silent", "--show-error", "--location", "https://sh.rustup.rs", "--output", str(installer)], check=True)
        subprocess.run(["sh", str(installer), "-y", "--profile", "minimal", "--default-toolchain", RUST_TOOLCHAIN], check=True)
else:
    subprocess.run(["rustup", "toolchain", "install", RUST_TOOLCHAIN, "--profile", "minimal"], check=True)
    subprocess.run(["rustup", "default", RUST_TOOLCHAIN], check=True)

for tool in ("rustc", "cargo"):
    executable = shutil.which(tool)
    if executable is None:
        raise RuntimeError(f"{tool} was not installed")
    result = subprocess.run([executable, "--version"], check=True, capture_output=True, text=True)
    print(result.stdout.strip())


rustc 1.90.0 (1159e78c4 2025-09-14)
cargo 1.90.0 (840b83a10 2025-07-30)


In [4]:
def execute_r(mode, timeout_seconds):
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    log_path = RUN_DIR / f"tentacles_{mode}_{stamp}.log"
    environment = os.environ.copy()
    environment["CMPS355_R_LIB"] = str(R_LIB)
    environment["CMPS355_TENTACLES_SOURCE"] = str(SOURCE_DIR)
    exit_code = None
    with log_path.open("w", encoding="utf-8") as output:
        proc = subprocess.Popen([RSCRIPT, str(SCRIPT), mode, SHA],
                                stdout=output, stderr=subprocess.STDOUT, env=environment)
        start = time.monotonic()
        last_cache = start
        last_progress = start
        try:
            while proc.poll() is None:
                time.sleep(10)
                now = time.monotonic()
                if mode == "install" and now - last_cache >= 480:
                    try:
                        save_r_cache()
                    except Exception as exc:
                        print(f"R cache warning: {exc}")
                    last_cache = time.monotonic()
                if now - last_progress >= 90:
                    elapsed = int(now - start)
                    print(f"{mode}: elapsed={elapsed}s, log={log_path.stat().st_size} bytes, "
                          f"packages={len(completed_packages())}", flush=True)
                    last_progress = now
                if now - start > timeout_seconds:
                    proc.kill()
                    proc.wait()
                    output.write("\nPROCESS_TIMEOUT\n")
                    exit_code = -1
                    break
        finally:
            if exit_code is None:
                exit_code = proc.wait()
    if mode == "install":
        try:
            save_r_cache()
        except Exception as exc:
            print(f"R cache warning: {exc}")
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    result = {
        "mode": mode,
        "exit_code": exit_code,
        "log": str(log_path.relative_to(ROOT)),
        "source_sha_confirmed": f"SOURCE_SHA={SHA}" in lines,
        "api_pass": "AUTHOR_API=PASS" in lines,
        "probe_pass": "PROBE=PASS" in lines,
        "example_pass": "AUTHOR_PREPROCESS_EXAMPLE=PASS" in lines,
        "example_unavailable": "AUTHOR_PREPROCESS_EXAMPLE=UNAVAILABLE" in lines,
    }
    print(f"{mode}: exit={exit_code}; {result['log']}")
    print("\n".join(lines[-25:]))
    return result

existing = execute_r("probe", 180) if LOCK_PATH.exists() or RESTORED_R_CACHE else None
if existing is not None and existing["exit_code"] == 0 and existing["source_sha_confirmed"] and existing["api_pass"]:
    installation = {"mode": "existing_verified", "exit_code": 0, "source_sha_confirmed": True}
    probe = existing
else:
    installation = execute_r("install", 7200)
    if installation["exit_code"] == 0 and installation["source_sha_confirmed"] and installation["api_pass"]:
        probe = execute_r("probe", 180)
    else:
        probe = {"mode": "probe", "exit_code": None, "skipped": "installation_failed"}


install: elapsed=90s, log=200732 bytes, packages=5
install: elapsed=180s, log=202449 bytes, packages=5
install: elapsed=270s, log=377511 bytes, packages=12
install: elapsed=360s, log=396282 bytes, packages=15
install: elapsed=450s, log=2348419 bytes, packages=17
Cached 18 package directories
install: elapsed=543s, log=2397466 bytes, packages=23
install: elapsed=633s, log=2419419 bytes, packages=27
install: elapsed=723s, log=2447108 bytes, packages=32
install: elapsed=813s, log=2454661 bytes, packages=36
install: elapsed=903s, log=2479836 bytes, packages=37
Cached 41 package directories
install: elapsed=1000s, log=2483008 bytes, packages=41
install: elapsed=1090s, log=2504194 bytes, packages=48
install: elapsed=1180s, log=2511940 bytes, packages=53
install: elapsed=1270s, log=2518322 bytes, packages=53
install: elapsed=1360s, log=2532161 bytes, packages=58
install: elapsed=1450s, log=2549725 bytes, packages=68
Cached 71 package directories
install: elapsed=1542s, log=2554566 bytes, pack

In [5]:
package_ok = (installation.get("exit_code") == 0
              and installation.get("source_sha_confirmed")
              and probe.get("exit_code") == 0
              and probe.get("source_sha_confirmed")
              and probe.get("api_pass")
              and probe.get("probe_pass"))
smoke = execute_r("smoke", 900) if package_ok else {"mode": "smoke", "exit_code": None, "skipped": "package_not_verified"}
if not package_ok:
    status = "FAIL_INSTALL_OR_API"
elif smoke.get("exit_code") != 0:
    status = "FAIL_AUTHOR_EXAMPLE"
elif smoke.get("example_pass") and smoke.get("source_sha_confirmed"):
    status = "PASS_AUTHOR_PREPROCESS_EXAMPLE"
elif smoke.get("example_unavailable"):
    status = "PASS_PACKAGE_API_ONLY_EXAMPLE_UNAVAILABLE"
else:
    status = "FAIL_UNEXPECTED_SMOKE_OUTPUT"

report = {
    "method": "TENTACLES",
    "repository": REPO,
    "commit": SHA,
    "r_script_sha256": hashlib.sha256(SCRIPT.read_bytes()).hexdigest(),
    "data": "author_example_or_none",
    "installation": installation,
    "probe": probe,
    "smoke": smoke,
    "status": status,
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
report_path = RUN_DIR / f"tentacles_report_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
if package_ok and not LOCK_PATH.exists():
    LOCK_PATH.parent.mkdir(parents=True, exist_ok=True)
    LOCK_PATH.write_text(json.dumps({"repository": REPO, "source_commit": SHA}, indent=2) + "\n", encoding="utf-8")
print("Report:", report_path.relative_to(ROOT))
print("Status:", status)
if status.startswith("FAIL"):
    raise RuntimeError("TENTACLES verification did not pass; inspect the saved R logs")

smoke: exit=0; runs/official_r/tentacles_smoke_20261107T042913796359Z.log
 [64] tibble_3.3.1          plyr_1.8.9            KEGGREST_1.52.2      
 [67] Biobase_2.72.0        withr_3.0.3           S7_0.2.2             
 [70] future_1.82.0         survival_3.8-12       heatmaply_1.6.0      
 [73] Biostrings_2.80.2     mclust_6.1.3          pillar_1.11.1        
 [76] MatrixGenerics_1.24.0 stats4_4.6.1          foreach_1.5.2        
 [79] plotly_4.12.1         generics_0.1.4        C50_0.2.0            
 [82] S4Vectors_0.50.3      ggplot2_4.0.3         scales_1.4.0         
 [85] globals_0.20.0        xtable_1.8-4          class_7.3-24         
 [88] glue_1.8.1            tools_4.6.1           dendextend_1.19.1    
 [91] NeuralNetTools_1.5.3  data.table_1.18.6.1   annotate_1.90.0      
 [94] gower_1.0.2           locfit_1.5-9.12       webshot_0.5.5        
 [97] registry_0.5-1        XML_3.99-0.25         mvtnorm_1.4-2        
[100] colino_0.0.1          grid_4.6.1            Boruta_10.0.